# Gemma 4 Developer Agent — Local Run

Local port of the Kaggle starter notebook. All `/kaggle/...` paths are replaced by the
`CONFIG` cell below. Set the paths there once, then run the cells top to bottom.

**Environment requirements (read first):**
- The `vllm` wheel in the wheelhouse is `manylinux_2_31_x86_64` → **Linux x86_64 + NVIDIA CUDA only**. Run this on your GPU machine (native Linux or WSL2). It will not install/run on native Windows.
- Use **Python 3.12** (the wheelhouse ships cp312 binary wheels).
- You need an NVIDIA GPU with enough VRAM for the chosen model + a 32k context KV cache.

## 0. Local Configuration (EDIT THIS CELL)

In [ ]:
from pathlib import Path

# =============================================================================
# EDIT THESE PATHS FOR YOUR MACHINE
# =============================================================================

# Root of this project (folder that contains 'gemma-4-developer-agent' and the wheelhouse).
# In a notebook there is no __file__, so set this explicitly on the GPU box, e.g.:
#   PROJECT_ROOT = Path('/home/you/SWE-Agent')
# By default we auto-detect by walking up from the kernel's CWD looking for the data folder.
def _find_project_root(start: Path) -> Path:
    for cand in [start, *start.parents]:
        if (cand / 'gemma-4-developer-agent').is_dir() and (cand / 'gemma-4-developer-agent-wheelhouse').is_dir():
            return cand
    return start

PROJECT_ROOT = _find_project_root(Path.cwd().resolve())

# The competition DATA directory (contains tasks.jsonl, snapshots/, graphs/, embeddings/, wheels/, sandbox/, sample_submission/).
DATA_DIR = PROJECT_ROOT / 'gemma-4-developer-agent'

# The wheelhouse folder with all the .whl files (swegemma, adk_submission, vllm, torch, ...).
WHEELHOUSE_DIR = PROJECT_ROOT / 'gemma-4-developer-agent-wheelhouse'

# A writable scratch dir (replaces /kaggle/working).
WORKING_DIR = PROJECT_ROOT / 'working'

# -----------------------------------------------------------------------------
# MODEL
# -----------------------------------------------------------------------------
# Which competition model to run. Smallest-first (fits a single consumer GPU best):
#   'gemma-4-e2b-it-qat-w4a16-ct'   (smallest, ~2-3 GB weights)   <- default, closest to a small local model
#   'gemma-4-e4b-it-qat-w4a16-ct'   (~3-4 GB)
#   'gemma-4-12b-it-qat-w4a16-ct'   (~6-8 GB)
#   'gemma-4-31b-it-qat-w4a16-ct'   (~16-18 GB, needs a big GPU / multi-GPU)
MODEL_NAME = 'gemma-4-e2b-it-qat-w4a16-ct'

# Local directory that holds the downloaded model weights for MODEL_NAME.
# Point this at wherever you download/unpack the model (an HF-format folder with
# config.json + *.safetensors + tokenizer files). Example layouts:
#   PROJECT_ROOT / 'models' / MODEL_NAME
#   Path('/home/you/models') / MODEL_NAME
MODEL_PATH = PROJECT_ROOT / 'models' / MODEL_NAME

# vLLM serving knobs for a single-GPU box. Lower gpu_memory_utilization if you hit OOM;
# lower MAX_MODEL_LEN (e.g. 8192) to shrink the KV cache on small-VRAM GPUs.
GPU_MEMORY_UTILIZATION = 0.90
MAX_MODEL_LEN = 32768

# How many sample tasks to actually evaluate (keep small for a first run).
NUM_SAMPLE_TASKS = 1

# =============================================================================
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print('PROJECT_ROOT   =', PROJECT_ROOT)
print('DATA_DIR       =', DATA_DIR, '(exists:', DATA_DIR.exists(), ')')
print('WHEELHOUSE_DIR =', WHEELHOUSE_DIR, '(exists:', WHEELHOUSE_DIR.exists(), ')')
print('WORKING_DIR    =', WORKING_DIR)
print('MODEL_NAME     =', MODEL_NAME)
print('MODEL_PATH     =', MODEL_PATH, '(exists:', MODEL_PATH.exists(), ')')

## 1. Environment Configuration and Package Installation

Installs the harness and all dependencies from the **local wheelhouse** (offline, no PyPI).

In [ ]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'
# NOTE: we intentionally do NOT set TRANSFORMERS_OFFLINE=1 here so you can download the
# model the first time if needed. Set it to '1' once weights are local to force offline.


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        'LiteLLM', 'LiteLLM Router', 'LiteLLM Proxy', 'litellm',
        'httpx', 'httpcore', 'openai', 'urllib3',
        'opentelemetry', 'opentelemetry.context',
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

assert WHEELHOUSE_DIR.exists(), f'Wheelhouse not found at {WHEELHOUSE_DIR}'

# Install every wheel in the local wheelhouse. --no-deps + --no-index keeps it fully offline;
# the wheelhouse is expected to be self-contained (all transitive deps included).
wheels = sorted(str(w) for w in WHEELHOUSE_DIR.glob('*.whl') if 'cutlass' not in w.name.lower())
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-index',
     f'--find-links={WHEELHOUSE_DIR}', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')

## 2. Dataset and Sample Submission

In [ ]:
import re
from swegemma.models import load_tasks

assert DATA_DIR.exists(), f'DATA_DIR not found: {DATA_DIR}'

# Copy sample_submission to a writable directory for customization and packaging
SAMPLE_SUBMISSION_SRC = DATA_DIR / 'sample_submission'
AGENT_DIR = WORKING_DIR / 'sample_submission'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
shutil.copytree(SAMPLE_SUBMISSION_SRC, AGENT_DIR)

# Rewrite the agent YAMLs to the chosen MODEL_NAME. The sample LoRA adapters are shaped for
# the 31B model, so when we pick a different base model we strip the adapters and the
# 'adapter:' lines (otherwise vLLM would reject the mismatched adapter shapes).
if MODEL_NAME != 'gemma-4-31b-it-qat-w4a16-ct':
    for yaml_path in [AGENT_DIR / 'agent.yaml', *sorted((AGENT_DIR / 'sub_agents').glob('*.yaml'))]:
        if yaml_path.is_file():
            text = yaml_path.read_text(encoding='utf-8')
            text = re.sub(r'^(\s*model:\s*).*$', rf'\g<1>{MODEL_NAME}', text, flags=re.MULTILINE)
            text = re.sub(r'^\s*adapter:\s*.*\n?', '', text, flags=re.MULTILINE)
            yaml_path.write_text(text, encoding='utf-8')
    shutil.rmtree(AGENT_DIR / 'adapters', ignore_errors=True)

# Load tasks from the local dataset
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks = load_tasks(TASKS_PATH)

print(f'Data directory: {DATA_DIR}')
print(f'Agent directory: {AGENT_DIR} (model={MODEL_NAME})')
print(f'Loaded {len(tasks)} tasks from {TASKS_PATH.name}')
for t in tasks[:5]:
    print(f'  - {t.instance_id} ({t.repo} @ {t.base_commit[:8]})')

## 3. Code Graph Functions (CPU-only, no model needed)

This cell exercises the pre-computed repo code graph + embeddings. It needs no GPU, so it's a
good first check that the harness imported and the data paths are correct.

In [ ]:
from swegemma import graph as sg

sample_task = tasks[0]
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')

repo_graph = sg.get_graph(
    repo_name=sample_task.repo,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
sample_nodes = list(repo_graph.nodes())
print(f'Graph for {sample_task.repo} ({sample_task.instance_id}):')
print(f'  Nodes: {repo_graph.number_of_nodes()}, Edges: {repo_graph.number_of_edges()}')

query_node = next((n for n, deg in repo_graph.degree() if deg >= 5), sample_nodes[0])
neighbors = sg.get_neighbor(node=query_node, graph=repo_graph, max_neighbors=10)
print(f'\nNeighbors of {query_node!r} (showing up to 5):')
for n in neighbors[:5]:
    print(f'  - {n}')

similar = sg.get_similar_nodes(
    node=query_node, repo_name=sample_task.repo, k=5, graph=repo_graph,
    graph_dir=GRAPH_DIR, embeddings_dir=EMBEDDINGS_DIR, base_commit=sample_task.base_commit,
)
print(f'\nTop similar nodes to {query_node!r}:')
for item in similar:
    print(f"  - {item['node_name']} (similarity={item['similarity']:.4f})")

focal_nodes = [query_node, *neighbors[:4]]
subgraph = sg.get_induced_subgraph(repo_graph, focal_nodes)
print(
    f'\nInduced subgraph over {len(focal_nodes)} focal nodes: '
    f'{subgraph.number_of_nodes()} nodes, {subgraph.number_of_edges()} edges'
)

## 4. Start vLLM Server (needs GPU)

Starts a local vLLM OpenAI-compatible server on `127.0.0.1:8000` serving the local model,
then registers it with the ADK model registry. `tensor_parallel_size` adapts to the number
of GPUs detected (1 on a single-GPU box).

In [ ]:
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from adk_submission.discovery import discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

litellm.drop_params = True

ALLOWED_MODEL_NAMES = frozenset({
    'gemma-4-31b-it-qat-w4a16-ct',
    'gemma-4-12b-it-qat-w4a16-ct',
    'gemma-4-e4b-it-qat-w4a16-ct',
    'gemma-4-e2b-it-qat-w4a16-ct',
})
# 12B needs an architecture override for vLLM; add others here only if required.
MODEL_VLLM_EXTRA_ARGS = {
    'gemma-4-12b-it-qat-w4a16-ct': (
        '--hf-overrides',
        '{"architectures": ["Gemma4ForCausalLM"], "model_type": "gemma4"}',
    ),
}
_SAFE_PATH_COMPONENT = re.compile(r'[A-Za-z0-9][A-Za-z0-9._-]*')
INFERENCE_API_KEY = os.environ.get('LOCAL_API_KEY', 'EMPTY')

declared_models = discover_declared_models(AGENT_DIR)
assert len(declared_models) == 1, f'Expected exactly 1 declared model, got: {sorted(declared_models)}'
declared_model = next(iter(declared_models))
assert declared_model in ALLOWED_MODEL_NAMES, f'Unsupported model {declared_model!r}. Allowed: {sorted(ALLOWED_MODEL_NAMES)}'

assert MODEL_PATH.exists(), (
    f'Model weights not found at {MODEL_PATH}.\n'
    f'Download the {declared_model} weights (HF format: config.json + *.safetensors + tokenizer)\n'
    f'into that folder, or edit MODEL_PATH in the CONFIG cell.'
)

adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
adapter_map = getattr(adapters, 'adapters', None) or {}
for adapter_name in adapter_map:
    assert _SAFE_PATH_COMPONENT.fullmatch(adapter_name), f'Unsafe adapter name {adapter_name!r}'
detected_ranks = [
    r for info in adapter_map.values() if (r := getattr(info, 'rank', None)) is not None and r > 0
]

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
assert gpu_count >= 1, 'No CUDA GPU detected. vLLM requires an NVIDIA GPU. Run this cell on your GPU machine.'
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
print(f'Detected {gpu_count} GPU(s) -> tensor_parallel_size={tp_size}')

if 'server_instance' in globals() and server_instance is not None:
    server_instance.stop()

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=MAX_MODEL_LEN,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    enable_auto_tool_choice=True,
    enable_lora=bool(adapter_map),
    max_loras=max(1, len(adapter_map)),
    max_lora_rank=max(detected_ranks) if detected_ranks else 128,
    scheduler_reserve_full_isl=False,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
    extra_args=MODEL_VLLM_EXTRA_ARGS.get(declared_model, ()),
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

models = server_instance.create_model_registry(
    aliases=[declared_model],
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)

## 5. Run Phase 1 Inference and Phase 2 Verification

Uses the `subprocess` sandbox backend (no Docker needed).

In [ ]:
import asyncio
import concurrent.futures
import pandas as pd
import yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator


def run_sync(coro_or_fn, *args, **kwargs):
    """Run an async coroutine synchronously inside a notebook event loop."""
    if callable(coro_or_fn) and not asyncio.iscoroutine(coro_or_fn):
        fn = lambda: coro_or_fn(*args, **kwargs)
    else:
        fn = lambda: coro_or_fn
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())


eval_config_file = next(
    (c for name in ('eval_config.yaml', 'eval_config.yml') if (c := AGENT_DIR / name).is_file()),
    None,
)
raw_eval_cfg = (
    yaml.safe_load(eval_config_file.read_text(encoding='utf-8')) or {}
    if eval_config_file is not None else {}
)
eval_section = raw_eval_cfg.get('evaluation', raw_eval_cfg)

timeout_seconds = int(eval_section.get('timeout_seconds', 300))
verification_timeout_seconds = int(eval_section.get('verification_timeout_seconds', 300))
max_tool_calls = int(eval_section.get('max_tool_calls', 100))
max_time_minutes = float(eval_section.get('max_time_minutes', 60.0))
turns_raw = eval_section.get('max_turns', eval_section.get('max_llm_calls'))
max_turns = int(turns_raw) if turns_raw is not None else None

SAMPLE_TASKS = tasks[:NUM_SAMPLE_TASKS]
limits, gen_constraints = build_submission_limits()

eval_config = EvalConfig(
    tasks_path=TASKS_PATH,
    snapshots_dir=DATA_DIR / 'snapshots',
    results_dir=WORKING_DIR / 'results',
    submission_dir=AGENT_DIR,
    models=models,
    sandbox='subprocess',
    timeout_seconds=timeout_seconds,
    verification_timeout_seconds=verification_timeout_seconds,
    max_time_minutes=max_time_minutes,
    max_tool_calls=max_tool_calls,
    max_turns=max_turns,
    limits=limits,
    generation_constraints=gen_constraints,
    adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=15, overlap_size=2, token_threshold=14336, event_retention_size=5,
    ),
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    wheels_dir=DATA_DIR / 'wheels',
    verbose=False,
)

evaluator = Evaluator(eval_config)
predictions = []

for idx, task in enumerate(SAMPLE_TASKS, start=1):
    print(f'[{idx}/{len(SAMPLE_TASKS)}] Evaluating {task.instance_id} ({task.repo})...')
    result = run_sync(
        evaluator.evaluate_task, task=task, task_index=idx, total_tasks=len(SAMPLE_TASKS),
    )
    patch = (result.agent_patch or '').strip()
    predictions.append({'id': task.instance_id, 'prediction': patch if patch else 'NO_PATCH'})
    print(
        f'  -> resolved={result.resolved}, exit_code={result.test_exit_code}, '
        f'patch_chars={len(patch)}, tool_calls={result.tool_calls}, '
        f'duration={result.duration_seconds:.1f}s'
    )

submission_df = pd.DataFrame(predictions, columns=['id', 'prediction'])
display(submission_df)

## 6. Package Submission Archive

In [ ]:
import zipfile
from adk_submission import validate_directory
from swegemma.config import ALLOWED_SUBMISSION_EXTENSIONS, MAX_SUBMISSION_SIZE_BYTES

for dirpath, dirnames, filenames in os.walk(AGENT_DIR, followlinks=False):
    parent = Path(dirpath)
    for name in (*dirnames, *filenames):
        assert not (parent / name).is_symlink(), f'Symlinks are not permitted: {(parent / name).relative_to(AGENT_DIR)}'
validate_directory(AGENT_DIR, limits=limits)

zip_base = WORKING_DIR / 'submission'
zip_path = Path(shutil.make_archive(str(zip_base), 'zip', root_dir=AGENT_DIR))

with zipfile.ZipFile(zip_path, 'r') as zf:
    infos = zf.infolist()
    total_size = sum(i.file_size for i in infos)
    assert total_size <= MAX_SUBMISSION_SIZE_BYTES, f'Archive exceeds 3 GiB limit: {total_size}'
    for info in infos:
        if not info.is_dir():
            ext = Path(info.filename).suffix.lower()
            assert ext in ALLOWED_SUBMISSION_EXTENSIONS, f'Disallowed file extension: {info.filename}'

print(f'Created {zip_path} ({zip_path.stat().st_size / (1024 * 1024):.2f} MiB)')